# Exploratory Data Analysis

This notebook is the first step in the Seattle building energy case study. It introduces the raw dataset, inspects column structure, evaluates missing values, and reviews the semantic role of the main fields before any cleaning or modeling.

## Objectives

- Inspect the raw Seattle energy dataset.
- Identify key columns for prediction and feature engineering.
- Check missing data, categorical variables, and compliance metadata.
- Save a filtered dataset for use in the preparation notebook.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import os
import numpy as np
from pathlib import Path

## Load the raw dataset

In [2]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

input_file = project_root / "data" / "raw" / "2016_Building_Energy_Benchmarking.csv"
df = pd.read_csv(input_file)

# Display column names to identify the usage column
print("Available columns:", df.columns.tolist())

print("Raw dataset loaded from:", input_file)
print("Dimensions:", df.shape)
print("Columns:", len(df.columns))

Available columns: ['OSEBuildingID', 'DataYear', 'BuildingType', 'PrimaryPropertyType', 'PropertyName', 'Address', 'City', 'State', 'ZipCode', 'TaxParcelIdentificationNumber', 'CouncilDistrictCode', 'Neighborhood', 'Latitude', 'Longitude', 'YearBuilt', 'NumberofBuildings', 'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking', 'PropertyGFABuilding(s)', 'ListOfAllPropertyUseTypes', 'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA', 'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA', 'YearsENERGYSTARCertified', 'ENERGYSTARScore', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)', 'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SteamUse(kBtu)', 'Electricity(kWh)', 'Electricity(kBtu)', 'NaturalGas(therms)', 'NaturalGas(kBtu)', 'DefaultData', 'Comments', 'ComplianceStatus', 'Outlier', 'TotalGHGEmissions', 'GHGEmissionsIntensity']
Raw dataset loaded from: C:\Users\ka

## Selection of Non-Residential Buildings

The goal is to restrict the dataset to building types relevant for modeling energy consumption in non-residential contexts and major campuses.

In [3]:
# Here are the values to keep:
non_residential_keywords = [
    "NonResidential",
    "Nonresidential COS",
    "Nonresidential WA",
    "SPS-District K-12",
    "Campus"
]

# Adapt here with the actual usage column name
candidate_columns = [
    "BuildingType",
    "PrimaryPropertyType",
    "LargestPropertyUseType",
    "ListOfAllPropertyUseTypes"
]
usage_column = next((col for col in candidate_columns if col in df.columns), None)

if usage_column is None:
    raise ValueError("No building type column found in the raw dataset.")

print("Using building type column:", usage_column)

# Filter rows containing the specified types
filtered_df = df[df[usage_column].isin(non_residential_keywords)]

print("Filtered rows:", filtered_df.shape[0], "/", df.shape[0])
print("Retained portion:", filtered_df.shape[0] / len(df))

Using building type column: BuildingType
Filtered rows: 1668 / 3376
Retained portion: 0.4940758293838863


In [4]:
output_file = project_root / "data" / "processed" / "2016_Building_Energy_Benchmarking_Purge.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save the result to a new CSV file
filtered_df.to_csv(output_file, index=False)

print(f"Filtered file saved to: {output_file}")

Filtered file saved to: C:\Users\karap\OpenClassRooms\dataprojet6\data\processed\2016_Building_Energy_Benchmarking_Purge.csv


## General Overview and Missing Values

Examine data types and detect columns with missing values before cleaning.

In [5]:
# 1. Loading data and general overview
purge_file = project_root / "data" / "processed" / "2016_Building_Energy_Benchmarking_Purge.csv"
building_consumption = pd.read_csv(purge_file)
# General overview
print("Dimensions:", building_consumption.shape)
building_consumption.head(5)

Dimensions: (1668, 46)


,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,...,Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,...,1.156514e+06,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,...,9.504252e+05,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,...,1.451544e+07,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,...,8.115253e+05,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,...,1.573449e+06,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88


### Data type info - null and non-null counts

In [6]:
# 2. General column information
# Info on data types and null/non-null counts
building_consumption.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1668 entries, 0 to 1667
Data columns (total 46 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    1668 non-null   int64  
 1   DataYear                         1668 non-null   int64  
 2   BuildingType                     1668 non-null   object 
 3   PrimaryPropertyType              1668 non-null   object 
 4   PropertyName                     1668 non-null   object 
 5   Address                          1668 non-null   object 
 6   City                             1668 non-null   object 
 7   State                            1668 non-null   object 
 8   ZipCode                          1652 non-null   float64
 9   TaxParcelIdentificationNumber    1668 non-null   object 
 10  CouncilDistrictCode              1668 non-null   int64  
 11  Neighborhood                     1668 non-null   object 
 12  Latitude            

In [7]:
# 3. Missing values analysis for building_consumption
# Count and percentage of missing values per column
missing_data = building_consumption.isnull().sum().to_frame('MissingCount')
missing_data['MissingPct'] = 100 * missing_data['MissingCount'] / len(building_consumption)
missing_data = missing_data[missing_data['MissingCount'] > 0].sort_values(by='MissingPct', ascending=False)

missing_data

,MissingCount,MissingPct
Comments,1668,100.000000
Outlier,1651,98.980815
YearsENERGYSTARCertified,1570,94.124700
ThirdLargestPropertyUseTypeGFA,1315,78.836930
ThirdLargestPropertyUseType,1315,78.836930
SecondLargestPropertyUseType,813,48.741007
SecondLargestPropertyUseTypeGFA,813,48.741007
ENERGYSTARScore,574,34.412470
ZipCode,16,0.959233
LargestPropertyUseTypeGFA,6,0.359712


### Categorical variable analysis

In [8]:
# 4. Categorical variable analysis

# Object-type columns (categorical or text)
cat_cols = building_consumption.select_dtypes(include='object').columns.tolist()

# Unique counts per categorical variable
building_consumption[cat_cols].nunique().sort_values(ascending=False)

PropertyName                     1664
Address                          1647
TaxParcelIdentificationNumber    1587
ListOfAllPropertyUseTypes         373
YearsENERGYSTARCertified           64
LargestPropertyUseType             56
SecondLargestPropertyUseType       47
ThirdLargestPropertyUseType        39
PrimaryPropertyType                22
Neighborhood                       19
BuildingType                        5
ComplianceStatus                    4
Outlier                             2
City                                1
State                               1
dtype: int64

## Semantic review of key columns

This table summarizes the main column groups and their likely role in the modeling pipeline.

| Column | Role | Notes |
| --- | --- | --- |
| `SiteEnergyUse(kBtu)` | Target variable | Main energy consumption target for regression. |
| `PropertyGFATotal` | Building size | Strong predictor of energy use. |
| `YearBuilt` | Building age | Useful as a physical descriptor. |
| `PrimaryPropertyType` | Building usage | Strong semantic feature. |
| `LargestPropertyUseType` | Usage category | May overlap with `PrimaryPropertyType` but still useful. |
| `ComplianceStatus` | Data quality filter | Use it to keep only reliable records. |
| `TotalGHGEmissions` | Emissions outcome | Related to energy use and can support domain analysis. |

## Compliance status analysis

Review the distribution of compliance statuses to decide whether to restrict the dataset to records with reliable reporting.

In [9]:
if 'ComplianceStatus' in filtered_df.columns:
    counts = filtered_df['ComplianceStatus'].value_counts(dropna=False)
    pct = filtered_df['ComplianceStatus'].value_counts(normalize=True, dropna=False) * 100
    compliance_summary = pd.DataFrame({
        'count': counts,
        'percent': pct.round(2)
    })
    display(compliance_summary)
else:
    print('ComplianceStatus column not found in this dataset.')

,count,percent
ComplianceStatus,,
Compliant,1548,92.81
Error - Correct Default Data,88,5.28
Non-Compliant,18,1.08
Missing Data,14,0.84


## Summary and next step

This exploratory notebook is meant to give students a first look at the raw dataset structure and the main semantic groups. The next notebook, `01_clean_data.ipynb`, uses this understanding to perform data cleaning, deduplication, and outlier detection before feature engineering.